In [2]:
import pandas as pd
import mysql.connector

In [3]:
conn=mysql.connector.connect(
    host="localhost",
    user="root",
    password="banumathi@#$0",
    database="revenue_prediction")
print('conect success')

conect success


In [14]:
query=""" SELECT *
FROM invoices
LIMIT 5;
"""

df = pd.read_sql(query, conn)
df.head()

C:\Users\ELCOT\AppData\Local\Temp\ipykernel_18816\789560219.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


,invoice_id,order_id,invoice_date,invoice_price
0,INV0001,ORD0001,2024-04-17,20542
1,INV0002,ORD0002,2024-11-09,71436
2,INV0003,ORD0003,2024-03-03,67882
3,INV0004,ORD0004,2024-02-05,18702
4,INV0005,ORD0005,2024-06-25,17241


In [16]:
query="""
SELECT
    so.order_id,
    so.product,
    so.list_price,
    i.invoice_price,
    i.invoice_date,
    dp.max_discount_pct
FROM sales_orders so
JOIN invoices i ON so.order_id = i.order_id
JOIN discount_policy dp ON so.product = dp.product
LIMIT 10;
"""
df=pd.read_sql(query,conn)
df.head()

C:\Users\ELCOT\AppData\Local\Temp\ipykernel_18816\673952441.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df=pd.read_sql(query,conn)


,order_id,product,list_price,invoice_price,invoice_date,max_discount_pct
0,ORD0001,Laptop,33843,20542,2024-04-17,16
1,ORD0002,Printer,55084,71436,2024-11-09,19
2,ORD0003,Tablet,78386,67882,2024-03-03,15
3,ORD0004,Printer,58784,18702,2024-02-05,19
4,ORD0005,Printer,47835,17241,2024-06-25,19


In [21]:
query="""
SELECT
    so.order_id,
    so.product,
    so.list_price,
    dp.max_discount_pct,
    i.invoice_price,
    ROUND(
        (so.list_price * (1 - dp.max_discount_pct / 100.0)) - i.invoice_price,
        2
    ) AS leakage_amount
FROM sales_orders so
JOIN invoices i ON so.order_id = i.order_id
JOIN discount_policy dp ON so.product = dp.product
WHERE
    (so.list_price * (1 - dp.max_discount_pct / 100.0)) > i.invoice_price;
"""
df=pd.read_sql(query,conn)
df.head()

C:\Users\ELCOT\AppData\Local\Temp\ipykernel_18816\402206579.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df=pd.read_sql(query,conn)


,order_id,product,list_price,max_discount_pct,invoice_price,leakage_amount
0,ORD0001,Laptop,33843,16,20542,7886.12
1,ORD0004,Printer,58784,19,18702,28913.04
2,ORD0005,Printer,47835,19,17241,21505.35
3,ORD0007,Tablet,66056,15,18916,37231.60
4,ORD0008,Laptop,77413,16,45701,19325.92


In [23]:
query="""
SELECT
    YEAR(i.invoice_date)  AS year,
    MONTH(i.invoice_date) AS month,
    ROUND(
        SUM(
            CASE
                WHEN (so.list_price * (1 - dp.max_discount_pct / 100)) - i.invoice_price > 0
                THEN (so.list_price * (1 - dp.max_discount_pct / 100)) - i.invoice_price
                ELSE 0
            END
        ),
        2
    ) AS total_leakage
FROM sales_orders so
JOIN invoices i
    ON so.order_id = i.order_id
JOIN discount_policy dp
    ON so.product = dp.product
GROUP BY
    YEAR(i.invoice_date),
    MONTH(i.invoice_date)
ORDER BY
    YEAR(i.invoice_date),
    MONTH(i.invoice_date);

"""
monthly = pd.read_sql(query, conn)
monthly.to_csv("monthly_price_leakage_trend.csv", index=False)
monthly.head()


C:\Users\ELCOT\AppData\Local\Temp\ipykernel_18816\2005044536.py:28: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  monthly = pd.read_sql(query, conn)


,year,month,total_leakage
0,2024,1,535472.89
1,2024,2,606095.83
2,2024,3,561477.13
3,2024,4,546831.95
4,2024,5,589715.61


In [22]:
# monthly.to_csv("monthly_price_leakage_trend.csv", index=False)
df.to_csv("order_level_leakage.csv", index=False)
